In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/llm-agentic-legal-information-retrieval/sample_submission.csv
/kaggle/input/llm-agentic-legal-information-retrieval/laws_de.csv
/kaggle/input/llm-agentic-legal-information-retrieval/val.csv
/kaggle/input/llm-agentic-legal-information-retrieval/court_considerations.csv
/kaggle/input/llm-agentic-legal-information-retrieval/train.csv
/kaggle/input/llm-agentic-legal-information-retrieval/test.csv


In [2]:
import pandas as pd
import numpy as np
import re
import os
import gc
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

# --- Config & Setup ---
class Cfg:
    input_path = '/kaggle/input/llm-agentic-legal-information-retrieval'
    top_k = 10  # Number of citations to retrieve per query
    use_semantic = True
    # Heuristic mapping for Swiss Law (En -> De acronyms)
    law_map = {
        'civil code': 'ZGB',
        'code of obligations': 'OR',
        'criminal code': 'StGB',
        'constitution': 'BV',
        'federal act': 'BG',
    }

print("Configuration set. Loading data...")

# --- Load Data ---
# Loading only necessary columns to save memory
laws = pd.read_csv(f'{Cfg.input_path}/laws_de.csv', usecols=['citation', 'text'])
test = pd.read_csv(f'{Cfg.input_path}/test.csv')
print(f"Laws shape: {laws.shape}")
print(f"Test query shape: {test.shape}")

# Optional: Load court cases if you have memory (commented out for speed/memory safety in demo)
# cases = pd.read_csv(f'{Cfg.input_path}/court_considerations.csv', usecols=['citation', 'text'])
# corpus = pd.concat([laws, cases], axis=0).reset_index(drop=True)
corpus = laws # Simple baseline uses only statutes

# --- Preprocessing ---
def clean_text(text):
    return str(text).lower().strip()

print("Preprocessing corpus...")
corpus['text_clean'] = corpus['text'].apply(clean_text)
corpus_citations = corpus['citation'].values

# --- 1. Semantic Retrieval Engine (TF-IDF Baseline) ---
# NOTE: For better score, replace TfidfVectorizer with 'sentence-transformers' 
# e.g. model.encode(queries) and model.encode(corpus)
if Cfg.use_semantic:
    print("Building TF-IDF Index...")
    vectorizer = TfidfVectorizer(
        max_features=25000, 
        stop_words='english', # rudimentary since queries are English
        ngram_range=(1, 2)
    )
    # Fit on both query and corpus to align vocab (imperfect for Cross-Lingual but runs)
    # Ideally: Translate queries to German first
    combined_text = pd.concat([test['query'], corpus['text_clean']])
    vectorizer.fit(combined_text)
    
    corpus_vecs = vectorizer.transform(corpus['text_clean'])
    print("Index built.")

# --- 2. Heuristic/Regex Engine ---
def extract_hard_matches(query):
    """
    Extracts patterns like 'Art. 12', 'Article 12', and maps 'Civil Code' -> 'ZGB'
    """
    query_lower = query.lower()
    citations = []
    
    # Check for law abbreviations
    detected_laws = []
    for en_term, de_abbr in Cfg.law_map.items():
        if en_term in query_lower:
            detected_laws.append(de_abbr)
    
    # Regex for Article numbers (e.g. "Art 12", "Article 55")
    # Matches: (Art|Article) (dot?) (space) (number)
    matches = re.findall(r'(?:art\.?|article)\s*(\d+[a-z]?)', query_lower)
    
    if matches and detected_laws:
        # Construct likely citations: "Art. {num} {law}"
        for num in matches:
            for law in detected_laws:
                citations.append(f"Art. {num} {law}")
    
    return citations

# --- Main Retrieval Loop ---
predictions = []

print("Starting inference...")
for idx, row in tqdm(test.iterrows(), total=len(test)):
    q_text = row['query']
    q_clean = clean_text(q_text)
    
    # A. Get Semantic Candidates
    candidates = []
    if Cfg.use_semantic:
        q_vec = vectorizer.transform([q_clean])
        # Compute similarity chunks to avoid OOM if corpus is huge
        scores = cosine_similarity(q_vec, corpus_vecs).flatten()
        # Get top indices
        top_indices = np.argpartition(scores, -Cfg.top_k)[-Cfg.top_k:]
        # Sort them by score descending
        top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]
        
        candidates = list(corpus.iloc[top_indices]['citation'].values)
        
    # B. Get Heuristic Candidates (High Precision)
    hard_matches = extract_hard_matches(q_text)
    
    # C. Ensemble: Prioritize Hard Matches, then Semantic
    # Note: We verify if hard matches actually exist in our corpus to avoid hallucinations
    valid_hard_matches = [c for c in hard_matches if c in set(corpus_citations)]
    
    final_set = list(dict.fromkeys(valid_hard_matches + candidates)) # Deduplicate preserve order
    
    # format as semicolon string
    pred_str = ";".join(final_set[:Cfg.top_k])
    predictions.append(pred_str)

# --- Save Submission ---
submission = pd.DataFrame({
    'query_id': test['query_id'],
    'predicted_citations': predictions
})

submission.to_csv('submission.csv', index=False)
print("Saved submission.csv")
print(submission.head())

# Cleanup
del vectorizer, corpus_vecs, corpus
gc.collect()

Configuration set. Loading data...
Laws shape: (175837, 2)
Test query shape: (40, 2)
Preprocessing corpus...
Building TF-IDF Index...
Index built.
Starting inference...


  0%|          | 0/40 [00:00<?, ?it/s]

Saved submission.csv
   query_id                                predicted_citations
0  test_001  Art. 48 Abs. 2 VID;Art. 9 Abs. 1 VATT;Art. 2 A...
1  test_002  Art. 1 Abs. 1 VRV;Art. 23 Abs. 1 VRV;Art. 76 A...
2  test_003  Art. 24 Abs. 3 412.101.220.19;Art. 114 Abs. 1 ...
3  test_004  Art. 108 Abs. 2 MepV;Art. 83 Abs. 4 MG;Art. 82...
4  test_005  Art. 12 Abs. 2 360.4;Art. 11 Abs. 1 360.4;Art....


19